# Normalização das bases

Padroniza endereços e atributos relacionados, filtra municípios de interesse e grava as bases intermediárias no formato silver.


In [1]:
# Objetivo: Carrega as bases e prepara as pastas de saida, limitando o processamento aos municipios consultados.

from pathlib import Path
import polars as pl

# Define os diretorios de entrada e saida e cria a pasta silver.
DATA_DIR = Path("../data/raw")
SILVER_DIR = Path("../data/silver")
SILVER_DIR.mkdir(parents=True, exist_ok=True)

# Mantem os valores originais como texto, inclusive marcadores como NULL.
# infer_schema=False: CNEFE tem colunas numericas com valor "NULL" como string
base = pl.scan_csv(DATA_DIR / "29_BA.csv", separator=";", infer_schema=False)
busca = pl.scan_csv(DATA_DIR / "2026_teste_tecnico_relinke_cidacs(in).csv", separator=";", infer_schema=False)

# Só processa municípios presentes na base de busca (o CNEFE da Bahia inteiro tem ~3,9M linhas)
municipios = busca.select("consulta_municipio").unique().collect().get_column("consulta_municipio").drop_nulls().to_list()
padrao_municipios = r"^(?:" + "|".join(municipios) + r")"

In [2]:
# Objetivo: Define tabelas auxiliares e funções de normalização para padronizar logradouros, numeros, CEP, bairros e complementos.

# ---------------------------------------------------------------------------
# Helpers de normalização
# ---------------------------------------------------------------------------

# Mapeia letras acentuadas para suas formas sem acento.
ACENTOS = (
    "ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ",
    "AAAAAEEEEIIIIOOOOOUUUUCN",
)

# Número por extenso -> digito. Compostos ("VINTE E CINCO") ANTES dos simples,
# senao "VINTE" e "CINCO" seriam substituidos separadamente.
# Converte numeros por extenso em digitos, priorizando formas compostas.
UNIDADES = {"UM": "1", "DOIS": "2", "TRES": "3", "QUATRO": "4", "CINCO": "5",
            "SEIS": "6", "SETE": "7", "OITO": "8", "NOVE": "9"}
DEZ_A_DEZENOVE = {"DEZ": "10", "ONZE": "11", "DOZE": "12", "TREZE": "13", "CATORZE": "14",
                  "QUATORZE": "14", "QUINZE": "15", "DEZESSEIS": "16", "DEZESSETE": "17",
                  "DEZOITO": "18", "DEZENOVE": "19"}
DEZENAS = {"VINTE": "20", "TRINTA": "30", "QUARENTA": "40", "CINQUENTA": "50"}

NUM_EXTENSO = {}
for dezena, vd in DEZENAS.items():
    for unidade, vu in UNIDADES.items():
        NUM_EXTENSO[f"{dezena} E {unidade}"] = str(int(vd) + int(vu))
NUM_EXTENSO.update(DEZ_A_DEZENOVE)
NUM_EXTENSO.update(DEZENAS)
NUM_EXTENSO.update(UNIDADES)

# Abreviações de tipo de logradouro -> forma completa (aplicado no inicio da string)
# Expande abreviacoes de tipos de logradouro para formas padronizadas.
TIPOS_ABREV = {
    "R": "RUA", "AV": "AVENIDA", "AVD": "AVENIDA", "AVDA": "AVENIDA", "AVEN": "AVENIDA",
    "TV": "TRAVESSA", "TRAV": "TRAVESSA", "TRV": "TRAVESSA",
    "AL": "ALAMEDA", "ALM": "ALAMEDA",
    "ESTR": "ESTRADA", "EST": "ESTRADA",
    "ROD": "RODOVIA", "RODOV": "RODOVIA",
    "PC": "PRACA", "PCA": "PRACA", "PR": "PRACA", "PRC": "PRACA",
    "LGO": "LARGO", "LG": "LARGO",
    "PQ": "PARQUE",
    "CJ": "CONJUNTO", "CONJ": "CONJUNTO", "CONJTO": "CONJUNTO",
    "JD": "JARDIM", "JARD": "JARDIM",
    "VL": "VILA", "VLA": "VILA",
    "CAM": "CAMINHO", "CAMI": "CAMINHO",
    "BEC": "BECO", "ESC": "ESCADARIA", "PSG": "PASSAGEM", "SRV": "SERVIDAO",
    "Q": "QUADRA", "QD": "QUADRA", "LT": "LOTE",
}

# Tipos completos, usados para montar o "core" (logradouro sem tipo)
# Lista tipos de via que podem ser removidos ao criar o nucleo do logradouro.
TIPOS_COMPLETOS = ["RODOVIA", "TRAVESSA", "AVENIDA", "ALAMEDA", "ESTRADA", "PRACA", "LARGO",
                   "PARQUE", "CONJUNTO", "LOTEAMENTO", "CONDOMINIO", "ESCADARIA", "PASSAGEM",
                   "SERVIDAO", "CAMINHO", "QUADRA", "JARDIM", "VILA", "DISTRITO", "FAZENDA",
                   "POVOADO", "COMUNIDADE", "SITIO", "BECO", "RUA"]

BAIRROS_GENERICOS = ["ZONA RURAL", "RURAL", "AREA RURAL", "ZONA URBANA", "URBANA", "SN", "S N", "CENTRO RURAL"]


# Define a normalização textual comum as colunas de endereco.
def norm_texto(col):
    """Caixa alta, sem acentos, sem pontuação, espaços colapsados."""
    c = col.fill_null("").str.strip_chars().str.to_uppercase()
    for origem, destino in zip(ACENTOS[0], ACENTOS[1]):
        c = c.str.replace_all(origem, destino)
    c = c.str.replace_all(r"""[.,;ºª'"!?()]""", "")
    c = c.str.replace_all(r"\s*-\s*", " ")
    c = c.str.replace_all(r"\s+", " ").str.strip_chars()
    return c


# Normaliza logradouros, abreviacoes, numeros por extenso e ruidos comuns.
def norm_logradouro(col):
    """Normalizacao especifica de logradouro para matching."""
    c = norm_texto(col)
    c = c.str.replace_all(r"^NONE\s+", "")                       # lixo "None FAZENDA X"
    c = c.str.replace_all(r"\s+S\s*/?\s*N$", "")                 # "RUA X S/N" no fim
    c = c.str.replace_all(r"(BR|BA|PE|RS|SP)(\d)", "${1} ${2}")  # BR101 -> BR 101
    for extenso, digito in NUM_EXTENSO.items():
        c = c.str.replace_all(rf"{extenso}", digito)
    for abrev, completo in TIPOS_ABREV.items():
        c = c.str.replace_all(rf"^{abrev}", completo)
    c = c.str.replace_all(r"\s+", " ").str.strip_chars()
    return c


# Remove o tipo da via para comparar apenas o nome principal do logradouro.
def logr_core(col_norm):
    """Logradouro sem o tipo (RUA/AVENIDA/...) - tolera divergencia de tipo entre as bases."""
    pat = "|".join(TIPOS_COMPLETOS)
    return col_norm.str.replace_all(rf"^(?:{pat})\s+", "")


# Retém CEPs com oito digitos e transforma os demais em valor nulo.
def cep_limpo(col):
    c = col.fill_null("").str.replace_all(r"\D", "")
    return pl.when(c.str.len_chars() == 8).then(c).otherwise(None)


# Limpa e extrai o número do endereço, tratando casos sem numeracao.
def numero_limpo(col):
    c = col.fill_null("").str.strip_chars().str.to_uppercase().str.replace_all(r"[\s.\-/]", "")
    c = pl.when((c == "") | c.str.starts_with("SN") | c.str.starts_with("SEM")).then(None).otherwise(c)
    return c.str.extract(r"(\d+)", 1)  # "102E" -> "102", "N332" -> "332"


# Normaliza complementos e remove indicações de ausencia de numero.
def comp_limpo(col):
    c = norm_texto(col)
    return pl.when((c == "") | c.str.contains(r"^(S\s?N|SEM( NUMERO)?)$")).then(None).otherwise(c)


# Normaliza bairros e remove classificacoes genericas sem valor discriminativo.
def bairro_norm(col):
    c = norm_texto(col)
    return pl.when((c == "") | c.is_in(BAIRROS_GENERICOS)).then(None).otherwise(c)

In [3]:
# Objetivo: Transforma os registros CNEFE para o formato silver e cria chaves auxiliares para localizar candidatos.

# ---------------------------------------------------------------------------
# CNEFE -> silver
# ---------------------------------------------------------------------------
# Combina os componentes de complemento disponiveis na base CNEFE.
comps = [pl.col(f"{p}_COMP_ELEM{i}") for i in range(1, 6) for p in ("NOM", "VAL")]
complemento = pl.concat_str(comps, separator=" ", ignore_nulls=True).str.strip_chars()

# Seleciona campos relevantes, aplica as funções de limpeza e cria chaves auxiliares.
base_silver = (
    base.filter(pl.col("COD_MUNICIPIO").str.contains(padrao_municipios))
    .select(
        "COD_UNICO_ENDERECO",
        "COD_MUNICIPIO",
        pl.col("COD_SETOR").str.strip_suffix("P").alias("COD_SETOR"),
        pl.col("CEP").alias("CEP"),
        pl.concat_str(["NOM_TIPO_SEGLOGR", "NOM_TITULO_SEGLOGR", "NOM_SEGLOGR"],
                      separator=" ", ignore_nulls=True)
            .str.replace_all(r"\s+", " ").str.strip_chars().alias("LOGRADOURO"),
        pl.when(pl.col("DSC_MODIFICADOR") == "SN")
          .then(None)
          .otherwise(numero_limpo(pl.col("NUM_ENDERECO"))).alias("NUMERO"),
        pl.col("DSC_LOCALIDADE").alias("LOCALIDADE"),
        complemento.alias("COMPLEMENTO"),
    )
    .with_columns(
        cep_limpo(pl.col("CEP")).alias("cep_limpo"),
        norm_logradouro(pl.col("LOGRADOURO")).alias("logr_norm"),
        bairro_norm(pl.col("LOCALIDADE")).alias("localidade_norm"),
        comp_limpo(pl.col("COMPLEMENTO")).alias("comp_norm"),
    )
    .with_columns(
    pl.col("cep_limpo").str.slice(0, 5).alias("cep5"),
    logr_core(pl.col("logr_norm")).alias("logr_core"),
)
.with_columns(
    pl.col("logr_core").str.slice(0, 4).alias("k_core"),
)
)

In [4]:
# Objetivo: Transforma a base de busca para o mesmo formato silver e grava as duas bases normalizadas em Parquet.

# ---------------------------------------------------------------------------
# Base de busca -> silver
# ---------------------------------------------------------------------------
# Padroniza os campos da busca segundo as mesmas regras usadas no CNEFE.
busca_silver = (
    busca
    .with_columns(
        pl.col("consulta_logradouro")
          .str.replace(r"(?i)^NONE\s+", "")
          .str.strip_chars().alias("consulta_logradouro"),
        numero_limpo(pl.col("consulta_numero")).alias("consulta_numero"),
        pl.when(pl.col("consulta_complemento").fill_null("").str.to_uppercase()
                    .str.contains(r"^\s*(S\s*/?\s*N|SEM\s*-?\s*N[UÚ]MERO?)\s*$"))
          .then(None)
          .otherwise(pl.col("consulta_complemento")).alias("consulta_complemento"),
    )
    .with_columns(
        norm_logradouro(pl.col("consulta_logradouro")).alias("logr_norm"),
        cep_limpo(pl.col("consulta_cep")).alias("cep_limpo"),
        bairro_norm(pl.col("consulta_bairro")).alias("bairro_norm"),
        comp_limpo(pl.col("consulta_complemento")).alias("comp_norm"),
    )
    .with_columns(
    pl.col("cep_limpo").str.slice(0, 5).alias("cep5"),
    logr_core(pl.col("logr_norm")).alias("logr_core"),
)
.with_columns(
    pl.col("logr_core").str.slice(0, 4).alias("k_core"),
)
)

# Grava as duas consultas lazy como arquivos Parquet na pasta silver.
base_silver.sink_parquet(SILVER_DIR / "cnefe_ba.parquet")
busca_silver.sink_parquet(SILVER_DIR / "busca.parquet")

print("Silver salva em:", SILVER_DIR)

Silver salva em: ..\data\silver


In [5]:
# Objetivo: Mostra amostras das bases silver e confere rapidamente o preenchimento das chaves de logradouro.

# Exibe amostras das duas bases gravadas para inspecao visual.
print("CNEFE Silver")
display(pl.scan_parquet(SILVER_DIR / "cnefe_ba.parquet").head(5).collect())

print("Busca Silver")
display(pl.scan_parquet(SILVER_DIR / "busca.parquet").head(5).collect())

# Confere se as chaves auxiliares estao preenchidas quando ha logradouro.
# checagem rapida: chaves de blocking nao podem ficar vazias quando o campo existe
check = pl.scan_parquet(SILVER_DIR / "cnefe_ba.parquet").select(
    pl.col("logr_norm").filter(pl.col("logr_norm") != "").len().alias("logr_norm_ok"),
    pl.col("k_core").filter(pl.col("logr_norm") != "").len().alias("k_core_ok"),
).collect()
print(check)

CNEFE Silver


COD_UNICO_ENDERECO,COD_MUNICIPIO,COD_SETOR,CEP,LOGRADOURO,NUMERO,LOCALIDADE,COMPLEMENTO,cep_limpo,logr_norm,localidade_norm,comp_norm,cep5,logr_core,k_core
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""192602843""","""2900702""","""290070220000001""","""48031000""","""RUA SEVERINO CALDEIRA""",null,"""RIACHO DA GUIA""","""""","""48031000""","""RUA SEVERINO CALDEIRA""","""RIACHO DA GUIA""",null,"""48031""","""SEVERINO CALDEIRA""","""SEVE"""
"""97526499""","""2900702""","""290070205000186""","""48024899""","""FAZENDA BURY""",null,"""FAZENDA JACARE""","""""","""48024899""","""FAZENDA BURY""","""FAZENDA JACARE""",null,"""48024""","""BURY""","""BURY"""
"""24004350""","""2900702""","""290070205000070""","""48009044""","""RUA JOSE JOAQUIM LEAL""","""346""","""KENNEDY""","""""","""48009044""","""RUA JOSE JOAQUIM LEAL""","""KENNEDY""",null,"""48009""","""JOSE JOAQUIM LEAL""","""JOSE"""
"""24004353""","""2900702""","""290070205000070""","""48009044""","""RUA JOSE JOAQUIM LEAL""",null,"""KENNEDY""","""""","""48009044""","""RUA JOSE JOAQUIM LEAL""","""KENNEDY""",null,"""48009""","""JOSE JOAQUIM LEAL""","""JOSE"""
"""192653750""","""2901353""","""290135305000008""","""48990000""","""FAZENDA ISAURA""",null,"""FAZENDA ISAURA""","""""","""48990000""","""FAZENDA ISAURA""","""FAZENDA ISAURA""",null,"""48990""","""ISAURA""","""ISAU"""


Busca Silver


consulta_logradouro,consulta_numero,consulta_bairro,consulta_cep,consulta_complemento,consulta_municipio,logr_norm,cep_limpo,bairro_norm,comp_norm,cep5,logr_core,k_core
str,str,str,str,str,str,str,str,str,str,str,str,str
"""RODOVIA BR 101""",null,"""BR101""","""48030260""",null,"""290070""","""RODOVIA BR 101""","""48030260""","""BR101""",null,"""48030""","""BR 101""","""BR 1"""
"""RUA RIO BAIANO""",null,"""CENTRO""","""46830000""",null,"""290130""","""RUA RIO BAIANO""","""46830000""","""CENTRO""",null,"""46830""","""RIO BAIANO""","""RIO """
"""FAZENDA BICO""",null,null,"""46130000""",null,"""290200""","""FAZENDA BICO""","""46130000""",null,null,"""46130""","""BICO""","""BICO"""
"""DISTRITO 2 DE JULHO""","""22""","""POCO CENTRAL""","""45675000""",null,"""290240""","""DISTRITO 2 DE JULHO""","""45675000""","""POCO CENTRAL""",null,"""45675""","""2 DE JULHO""","""2 DE"""
"""RUA SERGIO BOMFIM""","""332""","""BARREIRINHAS""","""47800020""",null,"""290320""","""RUA SERGIO BOMFIM""","""47800020""","""BARREIRINHAS""",null,"""47800""","""SERGIO BOMFIM""","""SERG"""


shape: (1, 2)
┌──────────────┬───────────┐
│ logr_norm_ok ┆ k_core_ok │
│ ---          ┆ ---       │
│ u32          ┆ u32       │
╞══════════════╪═══════════╡
│ 3889595      ┆ 3889595   │
└──────────────┴───────────┘
